# Instagram `description` 형태소 분석 및 A–H 목적 분류

이 노트북은 `instagram_normalized_3nf_posts.csv`의 `description`을 Kiwi로 형태소 분석하고, 아래 운영 기준에 따라 **규칙 기반 1차 분류**를 수행합니다. 결과는 정답 라벨이 아니라 수작업 검수의 출발점입니다. 특히 복합 목적 글은 `review_required=True`와 점수·근거어를 함께 확인하세요.

|코드|판정 목적|
|---|---|
|A|작품·캐릭터 감상 및 호불호|
|B|본인의 상품 관심·구매·소유·방문·요청·불만|
|C|팬 제작·꾸미기 콘텐츠|
|D|입고·판매·교환·공동구매·협찬·증정·팝업 안내|
|E|연예인·타 캐릭터·지인·반려동물 등 다른 대상 중심|
|F|번역·방영·정보·팬 교류·질문·안내|
|G|목적 판정 보류|
|H|외국어 중심 글|

In [ ]:
# 최초 1회만 실행하세요. 설치 후 커널 재시작이 필요할 수 있습니다.
%pip install -q pandas kiwipiepy tqdm matplotlib

In [ ]:
from pathlib import Path
from collections import Counter
import re

import matplotlib.pyplot as plt
import pandas as pd
from kiwipiepy import Kiwi
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 160)
tqdm.pandas()

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / '02_csv' / '02_instagram' / 'instagram_normalized_3nf_posts.csv').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 또는 입력 CSV를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_PATH = PROJECT_ROOT / '02_csv' / '02_instagram' / 'instagram_normalized_3nf_posts.csv'
OUTPUT_PATH = PROJECT_ROOT / '02_csv' / '02_instagram' / 'instagram_morphology_classified.csv'
REVIEW_PATH = PROJECT_ROOT / '02_csv' / '02_instagram' / 'instagram_morphology_review.csv'
FREQ_PATH = PROJECT_ROOT / '02_csv' / '02_instagram' / 'instagram_morpheme_frequency.csv'
INPUT_PATH

In [ ]:
df = pd.read_csv(
    INPUT_PATH,
    encoding='utf-8-sig',
    dtype={'post_id': 'string', 'description': 'string'},
    keep_default_na=False,
)
assert {'post_id', 'description'}.issubset(df.columns)
assert df['post_id'].is_unique, 'post_id 중복을 먼저 확인하세요.'
print(f'행: {len(df):,} / 빈 description: {df.description.str.strip().eq("").sum():,}')
df[['post_id', 'description']].head(3)

## 전처리와 형태소 분석

URL·계정 태그는 제거하고 해시태그의 `#`만 벗겨 분석합니다. 원문은 그대로 보존합니다. `content_tokens`는 분류에 유용한 체언·용언·부사·외국어 토큰이며, `morphemes`에는 조사·어미를 포함한 전체 `형태/품사`를 저장합니다.

In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+', re.I)
MENTION_RE = re.compile(r'(?<!\w)@[A-Za-z0-9_.]+')
SPACE_RE = re.compile(r'\s+')
HANGUL_RE = re.compile(r'[가-힣]')
FOREIGN_RE = re.compile(r'[A-Za-z一-龥ぁ-ゟ゠-ヿ]')

def normalize_text(text: str) -> str:
    text = URL_RE.sub(' ', str(text))
    text = MENTION_RE.sub(' ', text)
    text = text.replace('#', ' ')
    return SPACE_RE.sub(' ', text).strip()

kiwi = Kiwi(typos='basic')
for word in ['치이카와', '먼작귀', '하치와레', '우사기', '모몽가', '쿠리만쥬', '랏코', '시사', '카니']:
    kiwi.add_user_word(word, 'NNP')

KEEP_TAG_PREFIXES = ('NN', 'VV', 'VA', 'VX', 'MAG', 'MAJ', 'XR', 'SL', 'SH', 'SN')

def analyze(text: str) -> dict:
    clean = normalize_text(text)
    tokens = kiwi.tokenize(clean) if clean else []
    content = [t.form.lower() for t in tokens if t.tag.startswith(KEEP_TAG_PREFIXES) and len(t.form.strip()) > 1]
    return {
        'clean_description': clean,
        'morphemes': ' '.join(f'{t.form}/{t.tag}' for t in tokens),
        'content_tokens': ' '.join(content),
        'token_list': content,
    }

analysis = df['description'].progress_apply(analyze).apply(pd.Series)
work = pd.concat([df.copy(), analysis], axis=1)
work[['description', 'morphemes', 'content_tokens']].head(3)

## A–H 규칙 분류

문구 패턴은 3점, 형태소 키워드는 1점입니다. H는 문자 구성으로 먼저 분리하고, 나머지는 최고 점수 기준으로 정합니다. 무근거·이름 나열·짧은 글은 G로 보류합니다. `RULES`와 임계값은 검수 결과에 맞춰 조정하세요.

In [ ]:
CATEGORY_LABELS = {
    'A': '작품·캐릭터 감상', 'B': '상품 관심·구매·소유·방문', 'C': '팬 제작·꾸미기',
    'D': '판매·교환·협찬·팝업 안내', 'E': '다른 대상 중심', 'F': '정보·교류·질문',
    'G': '판정 보류', 'H': '외국어 중심',
}

RULES = {
 'A': {
   'terms': {'귀엽', '사랑', '좋아하', '최애', '입덕', '감동', '웃기', '재밌', '슬프', '목소리', '성격', '관계', '장면', '대사', '에피소드', '애니', '만화'},
   'phrases': [r'너무\s*귀여', r'진짜\s*(좋|싫)', r'최애', r'입덕', r'(장면|대사|목소리|성격).{0,12}(좋|귀엽|웃기|슬프)']},
 'B': {
   'terms': {'구매', '구입', '샀', '사다', '도착', '수령', '언박싱', '소장', '사용', '방문', '다녀오', '갖고싶', '사고싶', '품절', '불량', '환불', '교환', '선물받', '내돈내산', '굿즈'},
   'phrases': [r'(샀|구매했|구입했|주문했|도착했|수령했)', r'(사고|갖고)\s*싶', r'내돈내산', r'(매장|카페|팝업).{0,12}(방문|다녀)', r'(불량|환불|문의|아쉽)']},
 'C': {
   'terms': {'그림', '팬아트', '일러스트', '드로잉', '낙서', '더빙', '성대모사', '수공예', '뜨개질', '코스프레', '패러디', '꾸미기', '다꾸', '네일', '만들다', '제작'},
   'phrases': [r'(직접|제가|내가).{0,10}(그렸|만들|제작|더빙)', r'팬\s*아트', r'(그림|일러스트|낙서|드로잉)', r'(코스프레|패러디|다꾸|꾸미기|뜨개)']},
 'D': {
   'terms': {'판매', '입고', '재입고', '교환', '공구', '공동구매', '협찬', '광고', '증정', '나눔', '양도', '팝업', '오픈', '예약', '주문', '재고', '가격', '배송', '링크', '구매대행'},
   'phrases': [r'(판매|양도|교환|나눔)\s*(합니다|해요|중)', r'(입고|재입고|예약|주문)\s*(안내|오픈|시작)', r'(공동\s*구매|공구)\s*(모집|오픈)', r'(협찬|광고|제공받)', r'(가격|구매|주문).{0,15}(링크|디엠|DM|프로필)']},
 'E': {
   'terms': {'연예인', '아이돌', '배우', '가수', '멤버', '강아지', '고양이', '반려견', '반려묘', '친구', '지인', '조카', '아들', '딸', '닮은꼴', '산리오', '포켓몬', '농담곰'},
   'phrases': [r'(강아지|고양이|반려견|반려묘).{0,15}(닮|귀엽|입|장난감)', r'(아이돌|배우|가수|멤버).{0,15}(치이카와|굿즈|인형)', r'(친구|지인|조카|아들|딸).{0,12}(좋아|선물|닮)']},
 'F': {
   'terms': {'번역', '자막', '방영', '방송', '공개', '정보', '소식', '질문', '문의', '알려주', '어디', '언제', '누구', '친구', '맞팔', '소통', '교류', '모집', '공지', '안내', '정리'},
   'phrases': [r'(번역|자막|방영|방송|공개).{0,12}(정보|소식|일정|예정)', r'(어디|언제|누구|무엇|뭔가요|인가요|아나요|알려주세요)', r'(맞팔|소통|교류|친구).{0,10}(구해|찾아|모집)', r'(정보|공지|안내|정리)\s*(합니다|해요|입니다)']},
}

CHARACTER_ONLY = {'치이카와', '먼작귀', '하치와레', '우사기', '모몽가', '쿠리만쥬', '랏코', '시사', '카니'}
MIN_SCORE = 2

def foreign_dominant(text: str) -> tuple[bool, float, float]:
    compact = re.sub(r'[\W_\d]+', '', text)
    if not compact:
        return False, 0.0, 0.0
    hangul = len(HANGUL_RE.findall(compact))
    foreign = len(FOREIGN_RE.findall(compact))
    total = hangul + foreign
    return (foreign >= 8 and foreign / max(total, 1) >= 0.65), hangul / max(total, 1), foreign / max(total, 1)

def classify(row: pd.Series) -> pd.Series:
    text = row['clean_description']
    tokens = set(row['token_list'])
    is_foreign, ko_ratio, foreign_ratio = foreign_dominant(text)
    if is_foreign:
        return pd.Series({'category': 'H', 'category_label': CATEGORY_LABELS['H'], 'top_score': 0, 'second_score': 0, 'evidence': '문자비율', 'confidence': 'high', 'review_required': False, 'ko_ratio': ko_ratio, 'foreign_ratio': foreign_ratio})

    scores, evidence = {}, {}
    for code, rule in RULES.items():
        term_hits = sorted(tokens & rule['terms'])
        phrase_hits = [m.group(0) for p in rule['phrases'] if (m := re.search(p, text, re.I))]
        scores[code] = len(term_hits) + 3 * len(phrase_hits)
        evidence[code] = term_hits + phrase_hits

    ranked = sorted(scores, key=lambda c: (-scores[c], c))
    best, second = ranked[0], ranked[1]
    meaningful = tokens - CHARACTER_ONLY
    if not text or not meaningful or scores[best] < MIN_SCORE:
        category = 'G'
    else:
        category = best
    margin = scores[best] - scores[second]
    review = category == 'G' or (category not in {'G', 'H'} and (scores[best] < 4 or margin <= 1))
    confidence = 'low' if review else ('high' if scores[best] >= 7 and margin >= 3 else 'medium')
    return pd.Series({
        'category': category, 'category_label': CATEGORY_LABELS[category],
        'top_score': scores[best], 'second_score': scores[second],
        'evidence': ' | '.join(evidence[best]), 'confidence': confidence,
        'review_required': review, 'ko_ratio': ko_ratio, 'foreign_ratio': foreign_ratio,
    })

classified = work.progress_apply(classify, axis=1)
result = pd.concat([work.drop(columns=['token_list']), classified], axis=1)
result[['description', 'category', 'category_label', 'top_score', 'evidence', 'confidence', 'review_required']].head(10)

In [ ]:
summary = (result.groupby(['category', 'category_label'], observed=True)
           .agg(posts=('post_id', 'size'), review_posts=('review_required', 'sum'))
           .reset_index())
summary['share_pct'] = (summary['posts'] / len(result) * 100).round(2)
summary['review_pct'] = (summary['review_posts'] / summary['posts'] * 100).round(2)
display(summary.sort_values('category'))

ax = summary.set_index('category').reindex(list('ABCDEFGH'))['posts'].plot.bar(figsize=(10, 4), color='#6C8EBF')
ax.set(title='A–H 1차 분류 분포', xlabel='category', ylabel='posts')
plt.tight_layout()
plt.show()

In [ ]:
# 범주별 무작위 표본: 기준과 맞는지 먼저 검수하세요.
sample = (result.groupby('category', group_keys=False)
          .apply(lambda x: x.sample(min(5, len(x)), random_state=42), include_groups=False))
display(sample[['description', 'category_label', 'top_score', 'evidence', 'review_required']])

# 충돌·저신뢰 표본
display(result.loc[result['review_required'], ['post_id', 'description', 'category', 'top_score', 'second_score', 'evidence']]
        .sort_values(['category', 'top_score']).head(50))

In [ ]:
# 전체 형태소 빈도(분류용 품사만)
freq = Counter(token for text in result['content_tokens'] for token in text.split())
freq_df = pd.DataFrame(freq.most_common(), columns=['morpheme', 'count'])
display(freq_df.head(50))

# Excel 호환을 위해 UTF-8 BOM으로 저장
result.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
result.loc[result['review_required']].to_csv(REVIEW_PATH, index=False, encoding='utf-8-sig')
freq_df.to_csv(FREQ_PATH, index=False, encoding='utf-8-sig')
print(f'전체 결과: {OUTPUT_PATH}')
print(f'검수 대상: {REVIEW_PATH} ({result.review_required.sum():,}행)')
print(f'형태소 빈도: {FREQ_PATH}')

## 권장 검수 순서

1. 범주별 표본을 읽고 `RULES`의 누락어·오탐어를 조정합니다.
2. `review_required=True`를 사람이 우선 판정합니다.
3. 가능하면 범주별 100개 이상 수동 라벨을 만든 뒤 정확도·재현율을 측정합니다.
4. 한 글에 여러 목적이 강하면 단일 라벨 외에 보조 라벨 컬럼을 추가하는 방식을 고려합니다.

> 주의: B는 구매 관련 **게시물 수**이며 실제 구매자 수가 아닙니다. C는 공식 이용 허락 여부를 판단하지 않고, H는 국적·거주지를 뜻하지 않습니다.